In [10]:
import pandas as pd
import numpy as np
from pathlib import Path

import umap
import hdbscan
from sklearn.preprocessing import Normalizer
from sklearn.decomposition import PCA
from clustering.hdbscan_optimize import optimize_umap_hdbscan_auto
from visualization.samples_plot import show_clustering_samples
from visualization.plot import plot_clustering_on_umap
from hdbscan.validity import validity_index


PROJECT_ROOT = Path.cwd().resolve().parent
EMBEDDING_DIR = PROJECT_ROOT / "data" / "glomeruli" / "embeddings"

EMBEDDING_NAME = "densenet_crops_embeddings"

embedding: np.ndarray = np.load(EMBEDDING_DIR / f"{EMBEDDING_NAME}.npy")
csv = pd.read_csv(EMBEDDING_DIR / f"{EMBEDDING_NAME}.csv")

n_samples = embedding.shape[0]

In [11]:
embedding_l2 = Normalizer(norm="l2").fit_transform(embedding)

pca_95 = PCA(n_components=0.95)
pca_95_embedding = pca_95.fit_transform(embedding_l2)

In [ ]:
best = optimize_umap_hdbscan_auto(
    pca_95_embedding,
    n_runs=20,
    min_clusters=1,
    max_clusters=None,
    max_noise=0.35,
    min_valid_run_ratio=0.60,
    min_mean_ari=0.60,
    dbcv_weight=0.45,
    ari_weight=0.40,
    valid_run_ratio_weight=0.15,
    noise_weight=0.05,
    max_auto_param_combinations=80,
)

best_params = best["best_params"]
best_n_components = best["best_n_components"]

if best_n_components is None:
    raise ValueError("No suitable number of components found for UMAP.")

print(best_params)

In [ ]:
umap_params = best_params["umap"]

umap_embedding = umap.UMAP(
    n_neighbors=umap_params["n_neighbors"],
    min_dist=umap_params["min_dist"],
    n_components=umap_params["n_components"],
    metric=umap_params["metric"],
).fit_transform(pca_95_embedding)

In [ ]:
hdbscan_params = best_params["hdbscan"]

clusterer = hdbscan.HDBSCAN(
    min_cluster_size=hdbscan_params["min_cluster_size"],
    min_samples=hdbscan_params["min_samples"],
    metric=hdbscan_params["metric"],
)

labels = clusterer.fit_predict(umap_embedding)

probabilities = clusterer.probabilities_    # How much a points belong to the assigned cluster
outlier_scores = clusterer.outlier_scores_  # How much a point is anomalous compared to the rest of the data

In [ ]:
cluster_ids = sorted(label for label in np.unique(labels) if label != -1)

if len(cluster_ids) >= 2:
    dbcv_global, dbcv_per_cluster = validity_index(
        np.asarray(umap_embedding, dtype=np.float64),
        labels,
        metric="euclidean",
        per_cluster_scores=True,
    )
else:
    dbcv_global = np.nan
    dbcv_per_cluster = [np.nan for _ in cluster_ids]

dbcv_by_cluster = dict(zip(cluster_ids, dbcv_per_cluster))

cluster_dbcv_summary = pd.DataFrame(
    [
        {
            "cluster": int(cluster_id),
            "size": int(np.sum(labels == cluster_id)),
            "dbcv": float(dbcv_by_cluster[cluster_id]),
        }
        for cluster_id in cluster_ids
    ]
)

if np.any(labels == -1):
    cluster_dbcv_summary = pd.concat(
        [
            cluster_dbcv_summary,
            pd.DataFrame([
                {
                    "cluster": -1,
                    "size": int(np.sum(labels == -1)),
                    "dbcv": np.nan,
                }
            ]),
        ],
        ignore_index=True,
    )

print(f"DBCV globale: {dbcv_global:.4f}")
display(cluster_dbcv_summary.sort_values("cluster").reset_index(drop=True))

In [ ]:
plot_clustering_on_umap(
    pca_95_embedding,
    labels,
    n_neighbors=umap_params["n_neighbors"],
)

In [ ]:
image_paths = csv["image_path"].tolist()

sample_figure, sample_axes = show_clustering_samples(
    labels=labels,
    probabilities=clusterer.probabilities_,
    image_paths=image_paths,
    x=10,
    base_dir=PROJECT_ROOT,
    image_size=2.0,
)

In [ ]:
cols = [
    "n_components",
    "successful_runs",
    "valid_runs",
    "valid_run_ratio",
    "mean_dbcv",
    "mean_ari",
    "mean_dbcv_all",
    "mean_ari_all",
    "mean_n_clusters_all",
    "mean_noise_ratio_all",
    "valid_for_selection",
]

print(best["results"][cols].sort_values("n_components"))

print(
    best["run_details"]
    .groupby(["n_components", "constraint_reason"])
    .size()
    .unstack(fill_value=0)
)